# I'm looking at eval problems relating to elevation near mt san jacinto

https://inaturalist.slack.com/archives/C4GS9KY9J/p1753977149490269

Tony reports:

because it’s a taxon and area I’m somewhat familar with, an interesting test case would be the geomodel for Crotalus ruber. It’s very plentiful on Whitewater Canyon road, which is a famous herping road, but the geomodel doesn’t expect it there.

https://www.inaturalist.org/observations?nelat=34.07623818863033&nelng=-116.59320479407234&subview=map&swlat=33.870103054021854&swlng=-116.72916060461921&taxon_id=30724

https://www.inaturalist.org/geo_model/30724/explain

Spot checking the map, this map cell is being excluded from the evaluation here because the centroid is at very high elevation. Since we're now training with real locations, rather than centroids, the model doesn't learn that this snake can occur in grid cells with high elevation centroids.

I think if we eval using the true elevation for geo prior, we can get the model to make better predictions. However, for mapping purposes I'm not sure how to fix this. Moving down a grid cell size would help in this case, but for extreme variations I think this is going to happen at some resolution when there's extreme elevation changes, as long as we're discretizing for eval and not discretizing for train. 

For mapping purposes, we could take the average of the child cell elevations. 
Or we could evaluate on min and on max elevation of the children.
With the maps, we are making a statement that it does or does not occur in this grid cell - that shouldn't be based on the centroid but instead should be based on the characteristics of the grid cell - what geography is in it?

In [72]:
import h3
import numpy as np
import pandas as pd
import tensorflow as tf
import tifffile

from lib.models.geo_model_net import ResLayer
from lib.models.tf_gp_model import TFGeoPriorModel
from lib.models.encoders import CoordEncoder

In [2]:
target_taxon_id = 30724

In [6]:
tax = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20250420060005-aka-2.22/taxonomy.csv"
).set_index("taxon_id")
target_taxon = tax.loc[target_taxon_id]
target_taxon

parent_taxon_id            30692.0
rank_level                    10.0
leaf_class_id              69084.0
iconic_class_id                7.0
spatial_class_id             69084
name                Crotalus ruber
iconic_taxon_id            26036.0
rank                       species
Name: 30724, dtype: object

In [12]:
model_path_2_22 = "saved_models/saved_model_2_22_coords_elev.h5"
model_2_22 = TFGeoPriorModel(model_path_2_22)

# example
using https://www.inaturalist.org/observations/125587768 for example

In [16]:
lng = -116.65569
lat = 33.98925

In [26]:
h3_resolution = 4

In [33]:
cell_name = h3.geo_to_h3(lat, lng, h3_resolution)
cell_centroid = h3.h3_to_geo("8429a05ffffffff")
cell_centroid

(34.06261003531996, -116.7348885092861)

In [90]:
elevs = pd.read_csv("/home/alex/elevation_r4_5m.csv").set_index("h3_04")
elev_for_cell = elevs.loc[cell_name]
# this is the elevation we'll use at query time
print(f"The elevation for the centroid is: {elev_for_cell.elevation}")

The elevation for the centroid is: 1421.2142857142858


In [57]:
raster = np.load("/home/alex/sinr/data/env/elev_scaled.npy")
encoder = CoordEncoder(encoding_strategy="sinusoidal", raster=raster)

In [64]:
encoded_loc = encoder.encode(np.array([[lng, lat]]))
encoded_loc

array([[-0.8937186 ,  0.92704322, -0.44862797,  0.37495449,  0.10198744]])

In [68]:
elevation_normalized = encoded_loc[0][-1]
elevation_normalized

0.10198744435528384

In [93]:
# denormalize the elevation
im = tifffile.imread("/home/alex/sinr/data/env/wc2.1_5m_elev.tif")
im = im.astype(np.float64)
max_elev = np.max(im)
elev_denorm = elevation_normalized * max_elev
# this is the elevation we use at training time
print(f"The interpolated elevation is: {elev_denorm}")

The interpolated elevation is: 670.465459191636


#### now let's get the scores for the geo model at this location, but using both the interpolated and centroid elevations

In [100]:
# this is the geo score for this location with interpolated elevation, 670m
geo_score_interpolated_elev = model_2_22.predict(encoded_loc)[0][int(target_taxon.leaf_class_id)].numpy()
print(f"The geo score for {target_taxon['name']} at this location with interpolated elevation is: {geo_score_interpolated_elev}")

The geo score for Crotalus ruber at this location with interpolated elevation is: 0.7633542418479919


In [88]:
encoded_loc_centroid = np.array([[
    -0.8937186 ,  0.92704322, -0.44862797,  0.37495449, elev_for_cell.elevation / max_elev
]])
encoded_loc_centroid

array([[-0.8937186 ,  0.92704322, -0.44862797,  0.37495449,  0.21618714]])

In [101]:
# this is the geo score for this location with centroid elevation, 1421m
geo_score_centroid_elev = model_2_22.predict(encoded_loc_centroid)[0][int(target_taxon.leaf_class_id)]
print(f"The geo score for {target_taxon['name']} at this location with centroid elevation is: {geo_score_centroid_elev}")

The geo score for Crotalus ruber at this location with centroid elevation is: 0.16327272355556488
